# Chan Application — Cross-Sectional Mean Reversion

**Goal:** Reproduce and understand the simple mean-reverting strategy in Ernest P. Chan, *Quantitative Trading*, including the impact of transaction costs. This notebook contains its own imports; no `requirements.txt` is needed.

**Data:** `SPX_20071123.txt` (not bundled). Put the file in the same folder as this notebook or edit `DATA_PATH`. Rows should be trading dates; columns should be stock prices. The original book's reported Sharpe ratios are approximately **0.958 gross** and **−2.162 after costs** for its particular dataset and implementation; your results may differ.

**Important:** This is a research/educational replication, not a production execution model.

## 1. Imports and parameters
Run this cell whenever you restart the kernel. Libraries still need to be installed in the selected Python environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("SPX_20071123.txt")
START_DATE = "2006-01-01"
END_DATE = "2006-12-31"
ONE_WAY_COST = 0.0005  # 5 bps per dollar traded
TRADING_DAYS = 252

## 2. Load and inspect prices

The original data use dates like `20060103`. We parse these into a `DatetimeIndex`, sort them, and check the shape and missing observations. If your file has a different delimiter, adapt `read_csv`. **No data are silently forward-filled**.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Place SPX_20071123.txt beside this notebook "
        "or change DATA_PATH."
    )

prices = pd.read_csv(DATA_PATH, sep=r"\s+", engine="python")
if "Date" not in prices.columns:
    raise ValueError("Expected a 'Date' column. Inspect the input file format.")
prices["Date"] = pd.to_datetime(prices["Date"].astype(str), format="%Y%m%d")
prices = prices.set_index("Date").sort_index()
prices = prices.apply(pd.to_numeric, errors="coerce")

if not prices.index.is_unique:
    raise ValueError("Duplicate dates found")
print("Shape (days, stocks):", prices.shape)
print("Period:", prices.index.min(), "to", prices.index.max())
print("Missing price cells:", int(prices.isna().sum().sum()))
display(prices.head())

## 3. Compute returns and reversal signals

For stock \(i\) on day \(t\):

\[
r_{i,t}=P_{i,t}/P_{i,t-1}-1, \qquad
ar r_t=\operatorname{mean}_{i\in U_t} r_{i,t}.
\]

The **raw contrarian signal** is \(s_{i,t}=-(r_{i,t}-ar r_t)\). Normalize its gross absolute exposure to one:

\[
w_{i,t}=rac{s_{i,t}}{\sum_{j\in U_t}|s_{j,t}|}.
\]

A stock that outperformed the cross-sectional average receives a short signal; an underperformer receives a long signal. For complete observations, the weights sum to zero and the sum of absolute weights is one. The signal is known only *after* day \(t\)'s return is observed.

In [ ]:
# Modern Pandas: avoid implicit filling in pct_change
returns = prices.pct_change(fill_method=None)

cross_section_mean = returns.mean(axis=1)
raw_signal = -returns.sub(cross_section_mean, axis=0)
gross_signal = raw_signal.abs().sum(axis=1)

weights = raw_signal.div(gross_signal.replace(0, np.nan), axis=0)
# NaN means unavailable data or no signal; zero means no position.
weights = weights.fillna(0.0)

print("Sample weights:")
display(weights.iloc[1:6, :5])
print("Max gross weight:", weights.abs().sum(axis=1).max())
print("Max |net weight|:", weights.sum(axis=1).abs().max())

## 4. Gross daily PnL and Sharpe ratio

Use **yesterday's** weights with today's returns:

\[
R^{gross}_{p,t}=\sum_i w_{i,t-1}r_{i,t}.
\]

This `shift(1)` avoids using today's completed return to trade at yesterday's price. The annualized Sharpe estimate (without a risk-free adjustment, following this simplified dollar-neutral setup) is

\[
\widehat{SR}=\sqrt{252}\,rac{ar R}{s(R)}.
\]

For comparability, the original code uses `np.std` with `ddof=0`; below we show both population- and sample-standard-deviation versions. Missing asset returns are **ignored** in the dot product, following the original `nansum` approach; this is a simplifying assumption that must be revisited in a realistic backtest.

In [ ]:
gross_daily = (weights.shift(1) * returns).sum(axis=1, min_count=1)
mask = (gross_daily.index >= pd.Timestamp(START_DATE)) & (gross_daily.index <= pd.Timestamp(END_DATE))
gross_sample = gross_daily.loc[mask].dropna()

def sharpe(series, ddof=0):
    x = pd.Series(series).dropna()
    if len(x) < 2 or x.std(ddof=ddof) == 0:
        return np.nan
    return np.sqrt(TRADING_DAYS) * x.mean() / x.std(ddof=ddof)

print("Gross Sharpe (ddof=0):", sharpe(gross_sample, ddof=0))
print("Gross Sharpe (ddof=1):", sharpe(gross_sample, ddof=1))
print("Number of evaluated days:", len(gross_sample))

## 5. Transaction costs and turnover

Chan's simplified one-way trading cost is \(c=0.0005\), i.e. **5 basis points** per unit of traded notional. Approximate turnover from target weights as

\[
T_t=\sum_i|w_{i,t}-w_{i,t-1}|,\qquad C_t=c T_t.
\]

To keep the PnL and costs on the **same holding interval**, we assign the change from \(w_{t-1}\) to \(w_t\) to the *following* period's PnL. This is a timing convention, not a claim that same-close execution is possible.

**Caveat:** The exact trade notional should account for weight drift due to price moves; this simple `diff` approach is a first-order approximation. It also excludes borrowing, financing, market impact, and explicit initial liquidation costs. If a security has missing returns, the true executable PnL is not identifiable from this price table alone.

In [ ]:
# At the end of t-1, target weights change; model their cost with next day's PnL.
turnover_target = weights.diff().abs().sum(axis=1, min_count=1)
cost_for_holding_day = turnover_target.shift(1) * ONE_WAY_COST
net_daily = gross_daily - cost_for_holding_day

comparison = pd.DataFrame({
    "gross_return": gross_daily,
    "turnover": turnover_target.shift(1),
    "cost": cost_for_holding_day,
    "net_return": net_daily,
}).loc[START_DATE:END_DATE].dropna()

summary = pd.Series({
    "Gross Sharpe": sharpe(comparison["gross_return"]),
    "Net Sharpe": sharpe(comparison["net_return"]),
    "Mean daily gross return": comparison["gross_return"].mean(),
    "Mean daily net return": comparison["net_return"].mean(),
    "Mean daily turnover": comparison["turnover"].mean(),
    "Mean daily modeled cost": comparison["cost"].mean(),
})
display(summary.to_frame("Value"))

## 6. Visualize gross vs net performance

For an illustrative compounded equity curve, compute \(\prod_{s\le t}(1+R_s)\). This is meaningful only under the assumed fixed notional/capital convention; short selling and leverage can change actual account returns.

In [ ]:
growth = (1 + comparison[["gross_return", "net_return"]]).cumprod()
ax = growth.plot(figsize=(10, 5), title="Chan mean reversion: before and after modeled costs")
ax.set_ylabel("Growth of 1 unit")
ax.set_xlabel("Date")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Cost sensitivity

Test a few **predeclared** cost assumptions instead of tuning a cost to make the strategy look good. One basis point (bp) = 0.0001.

In [ ]:
cost_bps = [0, 5, 10, 20]
cost_results = []
for bps in cost_bps:
    pnl = comparison["gross_return"] - (bps / 10000) * comparison["turnover"]
    cost_results.append({"cost_bps": bps, "Sharpe": sharpe(pnl), "mean_daily_return": pnl.mean()})

display(pd.DataFrame(cost_results).set_index("cost_bps"))

## 8. Research questions and next steps

1. Why can the gross Sharpe be positive while the net Sharpe is negative? Quantify average gross return versus turnover × cost.
2. Is the signal still profitable if positions are held longer (e.g. weekly rather than daily)? Compare a **preselected** set of holding periods.
3. Compare 1-, 5-, and 20-day **past-return reversal** signals, with properly lagged positions.
4. Add out-of-sample tests and avoid choosing the best lookback using the test period.
5. Examine missing observations, delistings, survivorship bias, point-in-time universe membership, and transaction costs based on **drifted** holdings.

### Citation
Ernest P. Chan, *Quantitative Trading* (strategy adapted for educational replication). Clearly attribute the original idea and distinguish your own modifications in a public repository.